In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to C:\Users\Aarushi
[nltk_data]     Sachdeva\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Aarushi
[nltk_data]     Sachdeva\AppData\Roaming\nltk_data...


True

In [3]:
data = pd.read_csv("../data/processed/balanced_toxicity_dataset.csv")
print(data.shape)
data.head()

(73690, 2)


,text,label
0,"""\n""""On articles about topics with many fansit...",neutral
1,~I heard that Dick Gephardt was actually presi...,neutral
2,The magnesium stress connection doesn't have a...,neutral
3,This ain't no mf http://t.co/dOOeogiROg bitch ...,toxic
4,&#8220;@ZvckSlvt3r704: My sister shittin on yo...,toxic


In [4]:
data["label"] = data["label"].map({"neutral": 0,"toxic": 1})
data["label"].value_counts()

label
0    36845
1    36845
Name: count, dtype: int64

In [5]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [7]:
data["clean_text"] = data["text"].apply(clean_text)
data[["text", "clean_text"]].head()

,text,clean_text
0,"""\n""""On articles about topics with many fansit...",on articles about topics with many fansites in...
1,~I heard that Dick Gephardt was actually presi...,i heard that dick gephardt was actually presid...
2,The magnesium stress connection doesn't have a...,the magnesium stress connection doesnt have a ...
3,This ain't no mf http://t.co/dOOeogiROg bitch ...,this aint no mf bitch ass
4,&#8220;@ZvckSlvt3r704: My sister shittin on yo...,zvckslvtr my sister shittin on you hos once ag...


In [8]:
def preprocess_text(text):
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    return " ".join(tokens)

In [9]:
data["processed_text"] = data["clean_text"].apply(preprocess_text)
data[["clean_text", "processed_text"]].head()

,clean_text,processed_text
0,on articles about topics with many fansites in...,article topic many fansites including link one...
1,i heard that dick gephardt was actually presid...,heard dick gephardt actually president miami d...
2,the magnesium stress connection doesnt have a ...,magnesium stress connection doesnt reference c...
3,this aint no mf bitch ass,aint mf bitch as
4,zvckslvtr my sister shittin on you hos once ag...,zvckslvtr sister shittin ho vsamone wodeh bout...


In [10]:
X = data["processed_text"]
y = data["label"]
print(X.head())
print(y.head())

0    article topic many fansites including link one...
1    heard dick gephardt actually president miami d...
2    magnesium stress connection doesnt reference c...
3                                     aint mf bitch as
4    zvckslvtr sister shittin ho vsamone wodeh bout...
Name: processed_text, dtype: object
0    0
1    0
2    0
3    1
4    1
Name: label, dtype: int64


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print("training :", len(X_train))
print("testing :", len(X_test))

training : 58952
testing : 14738
